<a href="https://colab.research.google.com/github/yosie111/rag_shul/blob/main/add_breadcrumb_to_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adding Breadcrumb Fields to Shulchan Aruch JSON

**Input:**
- Hierarchical JSON file (simanim → seifim) — e.g. `shulchan_aruch.json`
- Text file with siman headings — `Shulchan_Aruch_Text_Headlines.txt`

**Output:**
- Same JSON with `hilchot_group` and `siman_sign` fields added to each siman.
- `text` field is **not** modified.


## 1. Upload Files

In [1]:
import json, re, os
from google.colab import files as colab_files

TXT_NAME = 'Shulchan_Aruch_Text_Headlines.txt'
JSON_NAME = 'shulchan_aruch.json'  # ← change according to your file

# Upload text file (headings)
if not os.path.exists(TXT_NAME):
    print('Upload the text file (siman headings):')
    uploaded = colab_files.upload()
    name = list(uploaded.keys())[0]
    if name != TXT_NAME:
        os.rename(name, TXT_NAME)

# Upload JSON file
if not os.path.exists(JSON_NAME):
    print('Upload the JSON file:')
    uploaded = colab_files.upload()
    name = list(uploaded.keys())[0]
    if name != JSON_NAME:
        os.rename(name, JSON_NAME)

print('✅ Files ready')


Upload the text file (siman headings):


Saving ‏‏Shulchan_Aruch_Text_Headlines.txt to ‏‏Shulchan_Aruch_Text_Headlines.txt
Upload the JSON file:


Saving shulchan_aruch.json to shulchan_aruch.json
✅ Files ready


## 2. Build Siman Mapping from Text File

In [2]:
with open(TXT_NAME, encoding='utf-8') as f:
    lines = f.readlines()

siman_map = {}       # siman_num (str) → {'hilchot_group': ..., 'siman_sign': ...}
current_group = None

for line in lines:
    line = line.strip()
    if not line:
        continue

    # Siman line with description: "סימן 1 - דין השכמת הבוקר. ובו 9 סעיפים:"
    m = re.match(r'^סימן\s+(\d+)\s*-\s*(.+?)\.?\s*ובו', line)
    if m:
        siman_map[m.group(1)] = {
            'hilchot_group': current_group or '',
            'siman_sign':    m.group(2).strip().rstrip('.')
        }
        continue

    # Siman without description: "סימן 625 - ובו סעיף אחד:"
    m2 = re.match(r'^סימן\s+(\d+)\s*-?\s*ובו', line)
    if m2:
        siman_map[m2.group(1)] = {
            'hilchot_group': current_group or '',
            'siman_sign':    current_group or ''   # fallback to group name
        }
        continue

    # Hilchot group heading: "הלכות ציצית"
    if re.match(r'^הלכות\s', line):
        current_group = line

print(f'✅ Mapped {len(siman_map)} simanim')
# Examples
for s in ['1', '8', '128', '625']:
    if s in siman_map:
        info = siman_map[s]
        print(f'  Siman {s}: [{info["hilchot_group"]} > {info["siman_sign"]}]')


✅ Mapped 697 simanim
  Siman 1: [הלכות הנהגת אדם בבוקר > דין השכמת הבוקר]
  Siman 8: [הלכות ציצית > הלכות ציצית ועטיפתו]
  Siman 128: [הלכות נשיאת כפים > דיני נשיאת כפים ואיזה דברים הפוסלים בכהן]
  Siman 625: [הלכות סוכה > ]


## 3. Load JSON, Add Fields, Save


In [3]:
with open(JSON_NAME, encoding='utf-8') as f:
    data = json.load(f)

missing_simanim = []
new_simanim = []

for siman_obj in data['simanim']:
    sn = str(siman_obj['siman'])
    info = siman_map.get(sn)

    if not info:
        missing_simanim.append(sn)
        hg, ss = '', ''
    else:
        hg = info['hilchot_group']
        ss = info['siman_sign']

    # Rebuild dict in exact field order
    new_simanim.append({
        'siman':         siman_obj['siman'],
        'hilchot_group': hg,
        'siman_sign':    ss,
        'seifim':        siman_obj['seifim'],
    })

data['simanim'] = new_simanim

print(f'✅ Updated {len(data["simanim"])} simanim')
if missing_simanim:
    print(f'⚠️  Simanim missing from mapping: {missing_simanim}')
else:
    print('✅ Full coverage — all simanim mapped successfully')

# Example
s = data['simanim'][0]
print(f'\nExample (siman {s["siman"]}):')
print(f'  hilchot_group: {s["hilchot_group"]}')
print(f'  siman_sign:    {s["siman_sign"]}')
print(f'  text (seif 1): {s["seifim"][0]["text"][:90]}...')


✅ Updated 697 simanim
✅ Full coverage — all simanim mapped successfully

Example (siman 1):
  hilchot_group: הלכות הנהגת אדם בבוקר
  siman_sign:    דין השכמת הבוקר
  text (seif 1): יתגבר כארי לעמוד בבוקר לעבודת בוראו שיהא הוא מעורר השחר...


## 4. Save and Download

In [4]:
# Output file name
base = os.path.splitext(JSON_NAME)[0]
OUT_NAME = f'{base}_with_breadcrumb.json'

with open(OUT_NAME, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f'✅ Saved: {OUT_NAME}')
print(f'   Size: {os.path.getsize(OUT_NAME)/1024/1024:.1f} MB')

# Download
try:
    colab_files.download(OUT_NAME)
except:
    print('File available at current path')


✅ Saved: shulchan_aruch_with_breadcrumb.json
   Size: 4.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>